<a href="https://www.kaggle.com/code/page0526/tcga-brca-survival-analysis?scriptVersionId=334535259" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Cancer Survival Prediction with Foundation Models and MIL

In this project you will build a model that predicts **relative patient risk** from whole-slide pathology images. Instead of processing billions of image pixels directly, we use patch embeddings produced by **UNI2-h**, then combine the patches with **gated attention multiple instance learning (MIL)**.

### Learning goals

By the end, you should be able to:

1. explain survival time, events, censoring, and risk;
2. describe how a pathology foundation model turns patches into embeddings;
3. explain why a whole slide is a *bag* of instances in MIL;
4. prepare patient-level data, train gated ABMIL with Cox loss, and evaluate it with the concordance index.

> **How to use this notebook:** read each explanation, predict what the next cell should do, then complete every cell marked **YOUR TURN**. Hints are included; solutions can be discussed after each checkpoint.


# 1. What is survival analysis?

Survival analysis studies **time until an event**. Here, the event is death and the time is measured from diagnosis. For a patient who is still alive at their last follow-up, the exact survival time is unknown; that observation is **right-censored**.

```mermaid
flowchart LR
    A[Diagnosis] --> B{Observed outcome}
    B -->|Death observed| C[Event = 1]
    B -->|Alive at last follow-up| D[Event = 0: censored]
```

Our neural network outputs a **risk score**, not a number of remaining days. A larger score means the model believes the event is likely to occur sooner. The Cox partial-likelihood loss lets us learn from both observed and censored patients.

**Think–pair–share:** Why would treating a censored patient's last follow-up day as their death day teach the model the wrong lesson?


In [ ]:
# YOUR TURN — represent two patients
# Patient A died after 420 days. Patient B was alive at the 700-day follow-up.
# Fill in the event indicators (1 = event observed, 0 = censored).
patient_a = {"time": 420, "event": ...}
patient_b = {"time": 700, "event": ...}

patient_a, patient_b


# 2. Install necessary packages

The notebook uses PyTorch for deep learning, `h5py` for UNI2-h feature files, `lifelines` for survival metrics, and Weights & Biases for optional experiment tracking. On Kaggle, run the install cell once per fresh session.


In [ ]:
!pip install -q lifelines timm==0.9.16 openslide-python


In [ ]:
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")


### Optional: experiment tracking

Never paste API keys into a shared notebook. Store `wandb_api_key` in Kaggle Secrets. If you do not want experiment tracking, set `USE_WANDB = False`.


In [ ]:
USE_WANDB = False

if USE_WANDB:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb_key = UserSecretsClient().get_secret("wandb_api_key")
    wandb.login(key=wandb_key)


# 3. Data preparation

## 3.1 From a whole-slide image to UNI2-h embeddings

A pathology slide is far too large to feed through a neural network at once. It is divided into small patches. **UNI2-h**, a foundation model pretrained on many pathology images, converts every patch into a 1,536-number feature vector.

```mermaid
flowchart LR
    A[Whole-slide image] --> B[Image patches]
    B --> C[UNI2-h foundation model]
    C --> D[1536-dimensional patch embeddings]
    D --> E[One patient bag]
```

The expensive foundation-model step has already been completed. Each `.h5` file contains `features` and patch `coords`. Several slides may belong to one patient, so we group files by the first three fields of the TCGA identifier.


In [ ]:
from pathlib import Path

embedding_dir = Path("/kaggle/input/datasets/page0526/brca-uni/TCGA-BRCA_IDC")

patient_to_slides = {}

for f in embedding_dir.glob("*.h5"):

    # TCGA-3C-AALI-01Z-00-DX1.xxx.h5
    slide_name = f.stem

    # first 3 fields
    patient = "-".join(slide_name.split("-")[:3])

    patient_to_slides.setdefault(patient, []).append(str(f))


In [ ]:
# Inspect one patient without printing every embedding value.
example_patient = next(iter(patient_to_slides))
example_path = patient_to_slides[example_patient][0]

with h5py.File(example_path, "r") as file:
    features = file["features"][:]
    coords = file["coords"][:]

print("Patient:", example_patient)
print("Feature shape:", features.shape)
print("Coordinate shape:", coords.shape)


## 3.2 Create patient-level survival labels

The clinical table has repeated rows, so we keep one row per patient. `event` is 1 only when death was observed. For censored patients, `time` comes from the last follow-up rather than days to death.


In [ ]:
# YOUR TURN — load the tab-separated clinical table.
# Hint: use pd.read_csv(..., sep="\t", low_memory=False)
clinical_path = "/kaggle/input/datasets/page0526/brca-uni/clinical.tsv"
clinical = ...
clinical.head()


In [ ]:
patient_df = clinical.drop_duplicates(subset="cases.submitter_id").copy()
import numpy as np

patient_df["event"] = (
    patient_df["demographic.vital_status"]
    .eq("Dead")
    .astype(int)
)

patient_df["time"] = np.where(
    patient_df["event"] == 1,
    patient_df["demographic.days_to_death"],
    patient_df["diagnoses.days_to_last_follow_up"]
)
label_lookup = (
    patient_df
    .set_index("cases.submitter_id")[["time", "event"]]
    .to_dict("index")
)


### Checkpoint

Before continuing, confirm that `time` is numeric, `event` contains only 0 and 1, and both event types are present. Data checks catch mistakes before a long GPU training run.


In [ ]:
print(patient_df[["time", "event"]].dtypes)
print(patient_df["event"].value_counts(dropna=False))
patient_df[["time", "event"]].describe()


In [ ]:
# YOUR TURN — join slide paths to survival labels.
# Complete the two missing values in the dictionary.
dataset = []

for patient_id, slide_paths in patient_to_slides.items():
    if patient_id not in label_lookup:
        continue

    label = label_lookup[patient_id]
    try:
        time = float(label["time"])
    except (ValueError, TypeError):
        continue

    dataset.append({
        "patient_id": patient_id,
        "slides": slide_paths,
        "time": ...,
        "event": ...,
    })

print(f"Usable patients: {len(dataset)}")
dataset[0]


In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    stratify=[d["event"] for d in dataset]
)


## 3.3 Build bags and batches

In MIL, a patient is a **bag** and each patch embedding is an **instance**. Bags contain different numbers of instances, so the collate function pads them to equal length and creates a Boolean mask. The model uses the mask to ignore padding.


In [ ]:
class SurvivalDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        feats = []

        for slide in sample["slides"]:
            with h5py.File(slide, "r") as f:
                # YOUR TURN — convert the NumPy feature array to float32 tensor.
                # Expected shape after indexing: (number_of_patches, 1536)
                x = ...
            feats.append(x)

        return {
            "features": torch.cat(feats, dim=0),
            "time": torch.tensor(sample["time"], dtype=torch.float32),
            "event": torch.tensor(sample["event"], dtype=torch.float32),
        }


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    feats = [b["features"] for b in batch]

    lengths = torch.tensor(
        [len(f) for f in feats]
    )

    padded = pad_sequence(
        feats,
        batch_first=True
    )

    max_len = padded.size(1)

    mask = (
        torch.arange(max_len)[None]
        < lengths[:,None]
    )

    return {

        "features": padded,

        "mask": mask,

        "time": torch.stack(
            [b["time"] for b in batch]
        ),

        "event": torch.stack(
            [b["event"] for b in batch]
        )
    }


In [ ]:
EPOCHS = 50
BATCH_SIZE = 4

train_loader = DataLoader(
    SurvivalDataset(train_data), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)
val_loader = DataLoader(
    SurvivalDataset(val_data), batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)


# 4. Model architecture

## Multiple instance learning and gated attention

We have patient-level outcomes but no label for each patch. MIL learns from this weak supervision. Gated attention assigns a learnable weight to every valid patch, then computes a weighted average:

$$a_i \propto \exp\left(w^T(\tanh(Vh_i) \odot \sigma(Uh_i))\right), \qquad M=\sum_i a_i h_i$$

Here, $h_i$ is one UNI2-h patch embedding, $a_i$ is its attention weight, and $M$ is the patient embedding. Attention can help us inspect what influenced the model, but it is not proof of biological causality.


In [ ]:
class GatedAttention(nn.Module):
    def __init__(self, in_dim=1536, hidden=256):
        super().__init__()
        self.V = nn.Linear(in_dim, hidden)
        self.U = nn.Linear(in_dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, x, mask):
        A_v = torch.tanh(self.V(x))
        A_u = torch.sigmoid(self.U(x))
        logits = self.w(A_v * A_u).squeeze(-1)

        # YOUR TURN — hide padded positions before softmax.
        # Hint: replace invalid logits with a very negative number.
        logits = logits.masked_fill(..., -1e9)
        attention = torch.softmax(logits, dim=1)

        patient_embedding = torch.bmm(
            attention.unsqueeze(1), x
        ).squeeze(1)
        return patient_embedding, attention


In [ ]:
class SurvivalMIL(nn.Module):

    def __init__(self):

        super().__init__()

        self.attention = GatedAttention()

        self.head = nn.Sequential(

            nn.Linear(1536,256),

            nn.ReLU(),

            nn.Dropout(0.25),

            nn.Linear(256,1)
        )

    def forward(self,x,mask):

        patient_embedding,attn = self.attention(
            x,
            mask
        )

        risk = self.head(
            patient_embedding
        ).squeeze(-1)

        return risk,attn


## Cox partial-likelihood loss

The loss compares observed-event patients with patients who were still at risk at that time. Sorting times from largest to smallest lets `logcumsumexp` construct the risk sets efficiently and stably.


In [ ]:
def cox_loss(risk, time, event):
    order = torch.argsort(time, descending=True)
    risk = risk[order]
    event = event[order]
    log_cumulative_risk = torch.logcumsumexp(risk, dim=0)

    # YOUR TURN — implement the negative Cox partial log-likelihood.
    # Only rows with event == 1 contribute directly to the numerator.
    loss = ...
    return loss.sum() / event.sum().clamp(min=1)


In [ ]:
model = SurvivalMIL().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


# 5. Training

Each epoch has two phases:

1. **Training:** forward pass → loss → gradients → optimizer step.
2. **Validation:** forward pass without gradients → loss and C-index.

We save the checkpoint with the highest validation C-index. For a classroom run, start with `EPOCHS = 2`; use more epochs only after the full pipeline works.


In [ ]:
from lifelines.utils import concordance_index
from tqdm.auto import tqdm

best_val_cindex = -np.inf

for epoch in range(EPOCHS):
    model.train()
    train_losses = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        x = batch["features"].to(device)
        mask = batch["mask"].to(device)
        time = batch["time"].to(device)
        event = batch["event"].to(device)

        optimizer.zero_grad(set_to_none=True)

        # YOUR TURN — complete the standard PyTorch training steps.
        risk, _ = ...
        loss = ...
        ...
        ...

        train_losses.append(loss.item())

    model.eval()
    val_risks, val_times, val_events = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["features"].to(device)
            mask = batch["mask"].to(device)
            risk, _ = model(x, mask)
            val_risks.extend(risk.cpu().numpy())
            val_times.extend(batch["time"].numpy())
            val_events.extend(batch["event"].numpy())

    val_cindex = concordance_index(val_times, -np.asarray(val_risks), val_events)
    print(f"train loss={np.mean(train_losses):.4f} | val C-index={val_cindex:.3f}")

    if val_cindex > best_val_cindex:
        best_val_cindex = val_cindex
        torch.save({"model_state_dict": model.state_dict(),
                    "val_cindex": val_cindex}, "best.pt")


### Training checkpoint

Explain these lines to a partner before moving on:

- Why call `optimizer.zero_grad()` every batch?
- Why call `loss.backward()` before `optimizer.step()`?
- Why use `model.eval()` and `torch.no_grad()` for validation?
- Why select a checkpoint using validation—not training—performance?


# 6. Evaluation and visualization

The **concordance index (C-index)** asks whether the model correctly ranks comparable patient pairs. A value near 0.5 is random ranking and 1.0 is perfect ranking. Always report performance on data that was not used to update model weights.


In [ ]:
checkpoint = torch.load("best.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

times, events, risks = [], [], []
with torch.no_grad():
    for batch in val_loader:
        risk, _ = model(
            batch["features"].to(device),
            batch["mask"].to(device),
        )
        risks.extend(risk.cpu().numpy())
        times.extend(batch["time"].numpy())
        events.extend(batch["event"].numpy())

# YOUR TURN — compute the validation C-index.
# Hint: higher risk means shorter survival, so negate the risk scores.
cindex = ...
print(f"Validation C-index: {cindex:.3f}")


## Visualize attention on a slide

The following optional extension maps patch attention back to slide coordinates. Bright regions received more model attention. Treat this as a hypothesis-generation tool: attention is resolution-dependent and does not establish what tissue feature caused the prediction.


In [ ]:
import cv2
import openslide
import matplotlib.pyplot as plt

# =====================================================
# Load features
# =====================================================

h5_path = "/kaggle/input/datasets/page0526/brca-uni/TCGA-BRCA_IDC/TCGA-3C-AALI-01Z-00-DX1.F6E9A5DF-D8FB-45CF-B4BD-C6B76294C291.h5"
wsi_path = "/kaggle/input/datasets/page0526/brca-uni/wsi/data/TCGA-3C-AALI-01Z-00-DX1/TCGA-3C-AALI-01Z-00-DX1.F6E9A5DF-D8FB-45CF-B4BD-C6B76294C291.svs"

with h5py.File(h5_path, "r") as f:
    features = f["features"][:].squeeze(0)
    coords = f["coords"][:].squeeze(0)

print(features.shape)
print(coords.shape)

# =====================================================
# Inference
# =====================================================

device = "cuda"

feats = torch.from_numpy(features).float().unsqueeze(0).to(device)

mask = torch.ones(
    (1, feats.shape[1]),
    dtype=torch.bool,
    device=device,
)

net = model.module if isinstance(model, torch.nn.DataParallel) else model
net.eval()

with torch.no_grad():
    risk, attention = net(feats, mask)

attention = attention.squeeze(0).cpu().numpy()

print(f"Risk: {risk.item():.4f}")
print(f"Attention sum: {attention.sum():.6f}")

# =====================================================
# Normalize for visualization ONLY
# =====================================================

att_vis = attention.copy()

att_vis -= att_vis.min()
att_vis /= (att_vis.max() + 1e-8)

# =====================================================
# Open WSI
# =====================================================

slide = openslide.OpenSlide(wsi_path)

thumb_size = 2048

thumbnail = np.array(
    slide.get_thumbnail((thumb_size, thumb_size))
)

thumb_h, thumb_w = thumbnail.shape[:2]

w0, h0 = slide.dimensions

sx = thumb_w / w0
sy = thumb_h / h0

# =====================================================
# Heatmap accumulation
# =====================================================

heatmap = np.zeros((thumb_h, thumb_w), dtype=np.float32)
counter = np.zeros((thumb_h, thumb_w), dtype=np.float32)

patch_size = 224

patch_w = max(1, round(patch_size * sx))
patch_h = max(1, round(patch_size * sy))

for (x, y), score in zip(coords, att_vis):

    xx = int(x * sx)
    yy = int(y * sy)

    x2 = min(xx + patch_w, thumb_w)
    y2 = min(yy + patch_h, thumb_h)

    heatmap[yy:y2, xx:x2] += score
    counter[yy:y2, xx:x2] += 1

counter[counter == 0] = 1

heatmap /= counter

# =====================================================
# Smooth
# =====================================================

heatmap = cv2.GaussianBlur(
    heatmap,
    (0, 0),
    sigmaX=12,
)

heatmap /= (heatmap.max() + 1e-8)

# =====================================================
# Color map
# =====================================================

heat_uint8 = np.uint8(255 * heatmap)

colored = cv2.applyColorMap(
    heat_uint8,
    cv2.COLORMAP_JET,
)

colored = cv2.cvtColor(
    colored,
    cv2.COLOR_BGR2RGB,
)

# =====================================================
# Overlay only on attended regions
# =====================================================

overlay = thumbnail.copy()

mask_overlay = heatmap > 0.05

overlay[mask_overlay] = (
    0.55 * thumbnail[mask_overlay]
    + 0.45 * colored[mask_overlay]
).astype(np.uint8)

# =====================================================
# Plot
# =====================================================

fig, ax = plt.subplots(
    1,
    3,
    figsize=(22, 8),
)

ax[0].imshow(thumbnail)
ax[0].set_title("WSI")
ax[0].axis("off")

im = ax[1].imshow(
    heatmap,
    cmap="jet",
)

ax[1].set_title("Attention")
ax[1].axis("off")

plt.colorbar(
    im,
    ax=ax[1],
    fraction=0.046,
    pad=0.04,
)

ax[2].imshow(overlay)
ax[2].set_title("Overlay")
ax[2].axis("off")

plt.tight_layout()
plt.show()


### Final reflection

1. What information did UNI2-h learn before this project began?
2. In this notebook, what are the bag, instances, and bag-level label?
3. How does censoring change the learning problem?
4. What does a C-index of 0.70 mean—and what does it *not* mean?
5. Name one source of data leakage or bias you would check before clinical use.


# 7. References

1. [Introduction to Survival Analysis — University of Waterloo](https://uwaterloo.ca/women-in-mathematics/sites/default/files/uploads/documents/w24winter_intro_to_survival_analysis.pdf)
2. [UNI: a universal pathology foundation model — Nature Medicine](https://www.nature.com/articles/s41591-024-02857-3)
3. [Towards large-scale training of pathology foundation models (UNI2-h)](https://arxiv.org/abs/2409.17817)
4. [Attention-based Deep Multiple Instance Learning](https://proceedings.mlr.press/v80/ilse18a.html)
5. [PyTorch computer-vision notebook used as a teaching-style reference](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/03_pytorch_computer_vision.ipynb)

> This notebook is for education and research, not clinical decision-making.
